In [1]:

import pandas as pd
import numpy as np

destination_master = pd.read_csv(
    "../data/processed/destination_master.csv"
)

destination_features = pd.read_csv(
    "../data/processed/destination_features.csv"
)

print("Destination master shape:", destination_master.shape)
print("Destination features shape:", destination_features.shape)

display(destination_features.head())


Destination master shape: (76, 12)
Destination features shape: (76, 15)


,Location_Name,Located_City,Location_Type,Average_Rating,Review_Count,Most_Common_Travel_Month,Destination_District,Destination_Province,Mapped_District,Geography_Notes,District_Key,Accommodation_Count,Rating_Score,Review_Popularity,Travel_Month_Category
0,Abhayagiri Dagaba,Anuradhapura,Religious Sites,4.30,96,2,NaN,North Central,Anuradhapura,NaN,anuradhapura,58,0.8250,0.260163,2
1,Ambewela Farms,Nuwara Eliya,Farms,3.90,136,1,NaN,Central,Nuwara Eliya,NaN,nuwara eliya,93,0.7250,0.368564,1
2,Ariyapala Mask Museum,Ambalangoda,Museums,3.87,77,2,Galle,Southern,Galle,NaN,galle,284,0.7175,0.208672,2
3,Arugam Bay,Arugam Bay,Beaches,4.06,187,7,NaN,Eastern,Ampara,NaN,ampara,62,0.7650,0.506775,7
4,Baker's Falls,Nuwara Eliya,Waterfalls,3.96,74,3,NaN,Uva,Nuwara Eliya,NaN,nuwara eliya,93,0.7400,0.200542,3


In [2]:

def recommend_destinations(
    preferred_type=None,
    min_rating=0,
    travel_month=None,
    top_n=5
):
    results = destination_features.copy()

    if preferred_type:
        results = results[
            results["Location_Type"].str.casefold()
            == preferred_type.casefold()
        ]

    results = results[
        results["Average_Rating"] >= min_rating
    ]

    if travel_month is not None:
        results = results[
            results["Most_Common_Travel_Month"] == int(travel_month)
        ]

    results["Recommendation_Score"] = (
        0.7 * results["Rating_Score"]
        + 0.3 * results["Review_Popularity"]
    )

    results = results.sort_values(
        "Recommendation_Score",
        ascending=False
    )

    return results.head(top_n).reset_index(drop=True)


In [4]:

test_recommendations = recommend_destinations(
    preferred_type="Beaches",
    min_rating=4.0,
    top_n=5
)

display(
    test_recommendations[
        [
            "Location_Name",
            "Average_Rating",
            "Review_Count",
            "Recommendation_Score"
        ]
    ]
)

print("Number of recommendations:", len(test_recommendations))


,Location_Name,Average_Rating,Review_Count,Recommendation_Score
0,Bentota Beach,4.59,233,0.817681
1,Passikudah Beach,4.56,160,0.753081
2,Mirissa Beach,4.27,217,0.748673
3,Nilaveli Beach,4.33,165,0.716896
4,Hikkaduwa Beach,4.02,213,0.701671


Number of recommendations: 5


In [5]:

# Define highly rated destinations as relevant for this baseline test
rating_threshold = 4.0
k = 5

relevant_destinations = destination_features[
    (destination_features["Location_Type"].str.casefold() == "beaches")
    & (destination_features["Average_Rating"] >= rating_threshold)
]

recommended_names = set(test_recommendations["Location_Name"])
relevant_names = set(relevant_destinations["Location_Name"])

relevant_recommended = recommended_names.intersection(relevant_names)

precision_at_5 = len(relevant_recommended) / k if k else 0

recall_at_5 = (
    len(relevant_recommended) / len(relevant_names)
    if relevant_names else 0
)

print("Relevant beach destinations:", len(relevant_names))
print("Relevant destinations in Top 5:", len(relevant_recommended))
print(f"Precision@5: {precision_at_5:.2f}")
print(f"Recall@5: {recall_at_5:.2f}")


Relevant beach destinations: 7
Relevant destinations in Top 5: 5
Precision@5: 1.00
Recall@5: 0.71


In [6]:

evaluation_summary = pd.DataFrame({
    "Metric": [
        "Relevant Beach Destinations",
        "Relevant Destinations in Top 5",
        "Precision@5",
        "Recall@5"
    ],
    "Value": [
        len(relevant_names),
        len(relevant_recommended),
        precision_at_5,
        recall_at_5
    ]
})

display(evaluation_summary)

evaluation_summary.to_csv(
    "../data/processed/destination_evaluation_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Evaluation summary saved successfully!")


,Metric,Value
0,Relevant Beach Destinations,7.000000
1,Relevant Destinations in Top 5,5.000000
2,Precision@5,1.000000
3,Recall@5,0.714286


Evaluation summary saved successfully!
